# Performance

<!-- Drafted with the help of an AI assistant (Claude, Anthropic), disclosed per course rules. -->

Running time, memory and output size of the parser on the whole corpus, from `data/m2/m2_benchmark_summary.csv`, `data/m2/m2_resource_timeseries.csv`, `data/m2/run_summary.json`, `data/m2/m2_sessions.jsonl` and `data/m2/m2_papers.csv`, produced by `python -m src.benchmark.m2_benchmark`.

:::{note}
**Testbed.** Google Colab, **CPU-only** (2 vCPUs, 13.6 GB RAM, Python 3.13, Linux), as the statement requires. The run parsed **all 20,000 assigned publications and their 26,491 versions** in a single session with 2 worker processes. The copy of the five crawl runs from Drive to the local disk (restore, with a SHA-256 check of every file) is setup and is not part of the measured time.
:::

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px

DATA = Path("../data/m2")
summary = pd.read_csv(DATA / "m2_benchmark_summary.csv")
series = pd.read_csv(DATA / "m2_resource_timeseries.csv")
papers = pd.read_csv(DATA / "m2_papers.csv", dtype={"arxiv_id": str})
run = json.loads((DATA / "run_summary.json").read_text())
env = json.loads((DATA / "environment.json").read_text())
sessions = [json.loads(line) for line in (DATA / "m2_sessions.jsonl").read_text().splitlines()]
audit = json.loads((DATA / "measurement_audit.json").read_text())
print("Machine:", env["cpu_count"], "vCPUs,", round(env["ram_total_mb"] / 1000, 1), "GB RAM,",
      env["platform"], ", Python", env["python"])
print("Sessions:", len(sessions), "| measurement complete:", audit["measurement_complete"],
      "| interrupted sessions:", audit["interrupted_sessions"])

## Summary

*End-to-end wall time* is measured around the whole benchmarked command, from the start of the process to its exit; *time per publication* is that wall time divided by the 20,000 publications. *Maximum RAM* is the largest sampled resident memory of the runner and its two worker processes together; the sampler looks every half second, so a shorter spike can be missed. *Input size* is the corpus read, *output size* the Parquet shards, and *disk max* the input plus the peak of the local output and work folders.

In [ ]:
summary.set_index('metric')

## Where the wall time goes

`run_summary.json` times the parse loop alone, whereas the benchmark times the whole command. The difference is time outside the parse loop, including setup and finalization. Before parsing, the command verifies crawl journals, paper/version mapping and the SHA-256 of recovered `.bbl` files. The available measurements do not attribute the entire difference to a single stage; it is shown separately, not removed.

In [ ]:
total = sessions[0]["wall_s"]
loop = run["wall_s"]
pd.DataFrame(
    {
        "Seconds": [total, loop, total - loop],
        "Minutes": [total / 60, loop / 60, (total - loop) / 60],
        "Per publication (s)": [total / 20000, loop / 20000, (total - loop) / 20000],
    },
    index=["Benchmarked command, end to end", "Parse loop alone", "Startup and input verification"],
).round(3)

## RAM and disk over time

In [ ]:
sampled = series.iloc[::6]  # one point every 3 s keeps the page small
long = sampled.melt(id_vars=["session_id", "t_sec"], var_name="measure", value_name="MB")
long["measure"] = long["measure"].map(
    {"rss_total_mb": "RAM (runner + workers)", "disk_mb": "Local output + work folders"}
)
fig = px.line(
    long,
    x=long["t_sec"] / 60,
    y="MB",
    color="measure",
    labels={"x": "Minutes since the start of the command", "MB": "MB"},
    title="Resident memory of the process tree and size of the local output",
)
fig

## Where the time goes per publication

`elapsed_s` is the time one worker spent on one publication (all its versions, from reading the files to the rows of the three tables).

In [ ]:
fig = px.histogram(
    papers,
    x="elapsed_s",
    nbins=60,
    log_y=True,
    labels={"elapsed_s": "Seconds per publication"},
    title="Time spent on one publication",
)
fig.update_layout(yaxis_title="Publications (log scale)")
fig

In [ ]:
papers["elapsed_s"].describe(percentiles=[0.5, 0.9, 0.99, 0.999]).round(3).to_frame("seconds")

In [ ]:
papers.sort_values("elapsed_s", ascending=False).head(10)[
    ["arxiv_id", "n_versions", "elements_before", "bibtex_before", "elapsed_s"]
]

## What these numbers mean

- One session on one Colab runtime: they describe that runtime and that day, not a range. No run was interrupted, so no session had to be added up.
- The time per publication is a mean over a corpus in which 14,913 of the 20,000 publications have a single version; a publication with many versions costs more (see the slowest ones above).
- Memory stays under 1 GB for the whole tree of processes because a worker handles one publication at a time and results are written in shards of 100 publications, not for the whole corpus at once.
- While the parser ran, the notebook also copied the finished shards to Drive. That copy competes for the disk and the network, so the wall time is that of a run that saves its state as it goes.